## Problem Statement

Frankfurt am Main has several years of recorded traffic accident data. Within that city, certain locations are not merely "unlucky once" — they exhibit elevated risk consistently, year after year. Other locations may show an elevated accident count in a single year purely by chance. The objective of this session is to separate **signal** (persistent, structural risk) from **noise** (a one-off cluster of unrelated events).

A naive approach — counting accidents per location and ranking the results — is insufficient for two reasons:

1. **Random clustering occurs even in uniform risk distributions.** Without a statistical test, apparent clusters cannot be distinguished from chance occurrences.
2. **A single year of elevated risk is not sufficient evidence.** A location that was elevated once across several years represents a materially different situation than one that was elevated in every year observed.

The approach therefore applies a recognized spatial statistic independently to each year of data, then combines the yearly results, and retains only what is corroborated across the observed period.

### Why Frankfurt

The showcase uses a real municipal dataset, for a real city, spanning a realistic multi-year window. This reflects the kind of practical data engineering challenge enterprise GIS teams encounter in production, rather than a simplified or synthetic example.

## Key Concepts

### Hot Spot Analysis (Getis-Ord Gi*)

Hot Spot Analysis is a recognized spatial statistic. Observations (in this case, accidents) are aggregated into spatial units, and for each unit, the Getis-Ord Gi* statistic evaluates whether the observed count — considered together with its neighbors — is higher than would be expected under a random distribution. Only when this exceeds a statistical confidence threshold is a unit classified as a hot spot.

The result is expressed as an ordinal confidence field, conventionally named `Gi_Bin`, ranging from **-3 to +3**:

| Gi_Bin | Meaning |
|---|---|
| **+3** | Hot spot, 99% confidence |
| **+2** | Hot spot, 95% confidence |
| **+1** | Hot spot, 90% confidence |
| **0** | Not statistically significant |
| **-1 / -2 / -3** | Cold spot (statistically fewer observations than expected), mirrored confidence levels |

Cold spots are a legitimate and interesting finding in their own right — for example, they may indicate the effect of a previous safety intervention — but this showcase focuses on the hot side of the spectrum: where risk persists.

Read more: [Hot Spot Analysis Tool](https://doc.esri.com/en/arcgis-pro/latest/tool-reference/spatial-statistics/h-how-hot-spot-analysis-getis-ord-gi-spatial-stati.html#0BE)

In [1]:
%load_ext dotenv

In [4]:
%dotenv

In [48]:
from arcgis.gis import GIS
from arcgis.map import Map
import dotenv
import os
import pandas as pd
from traffic_safety.utils import combine_yearly_hotspots, list_feature_classes, summarize_hotspot_bins, generate_hotspots_renderer, filter_hotspot_bins, publish_hotspot_layer

### The Alignment Problem

Each year's Hot Spot Analysis produces its own independent spatial grid of analysis units. Grids produced in different years are not guaranteed to be identical in extent, orientation, or boundary placement. As a result, units cannot be joined across years by a shared identifier — there is no common key.

The approach adopted here designates one year as a spatial reference. For every analysis unit in that reference year, a representative point (the unit's centroid) is computed. For every other year, the analysis unit containing that point is identified, and its confidence value is associated with the reference unit. Repeating this across all available years produces a single table in which every row represents one spatially consistent location, with one confidence value contributed by each year — despite the underlying grids never being identical.

In [40]:
home_portal = GIS("home")

In [9]:
gdb_path = os.getenv("HOTSPOT_GDB")
feature_classes = list_feature_classes(gdb_path)
feature_classes

['Traffic_Accidents_2019_Projected_HotSpots',
 'Traffic_Accidents_2020_Projected_HotSpots',
 'Traffic_Accidents_2021_Projected_HotSpots',
 'Traffic_Accidents_2022_Projected_HotSpots',
 'Traffic_Accidents_2023_Projected_HotSpots',
 'Traffic_Accidents_2024_Projected_HotSpots',
 'Traffic_Projected_HotSpots']

In [46]:
from ipywidgets import HBox, VBox, Label, Layout
from traffic_safety.utils import generate_hotspots_renderer, read_feature_classes

def show_hotspot_map(portal: GIS, df: pd.DataFrame):
    basemap_item_id = os.getenv("CUSTOM_BASEMAP_ITEM_ID")
    if basemap_item_id:
        portal_item = portal.content.get(basemap_item_id)
        map_view = Map(portal_item)
    else:
        map_view = portal.map("Germany")
        map_view.basemap.basemap = "osm"
    drawing_info = { 
        "renderer": generate_hotspots_renderer()
    }
    fset = df.spatial.to_featureset()
    map_view.content.add(fset, drawing_info=drawing_info, options={"opacity": 0.7})
    map_view.zoom_to_layer(fset)
    return map_view

In [17]:
hotspots_dict = read_feature_classes(gdb_path, [feature_classes[0], feature_classes[-1]])
hotspots_2019_df, hotspots_2025_df = hotspots_dict.values()

In [28]:
left_map = show_hotspot_map(home_portal, hotspots_2019_df)
right_map = show_hotspot_map(home_portal, hotspots_2025_df)

In [29]:
def side_by_side(left_map, left_label, right_map, right_label):
    # sync maps' navigation
    #left_map.sync_navigation(right_map)

    """
    def sync_navigation(change):
        if right_map.center != left_map.center:
            right_map.center = left_map.center
    
        if right_map.zoom != left_map.zoom:
            right_map.zoom = left_map.zoom

    left_map.observe(sync_navigation, names=["center", "zoom"])
    """

    # Create VBoxes for each map and label
    box1 = VBox([Label(left_label), left_map], layout=Layout(width='50%'))
    box2 = VBox([Label(right_label), right_map], layout=Layout(width='50%'))

    # Place the VBoxes side by side using an HBox
    hbox = HBox([box1, box2])

    # Display the HBox
    display(hbox)

In [30]:
side_by_side(left_map, "2019", right_map, "2025")

In [31]:
left_map.sync_navigation(right_map)

### Defining Persistence

Once aligned, each location carries up to one confidence value per year (a location may fall outside the analysis extent in a given year, in which case that year contributes no value). These yearly values are reduced to a small set of summary measures:

- **A central tendency measure (median)** — representative of the typical confidence level across the years in which the location was observed. A median is preferred over a mean because the confidence field is ordinal rather than strictly continuous; a median remains interpretable, whereas a mean can produce a value that does not correspond to any meaningful confidence level.
- **Minimum and maximum observed values** — characterizing the weakest and strongest year for that location.
- **A count of years observed** — the number of years that actually contributed a value for that location. Years in which a location falls outside the study extent are excluded from the calculation rather than treated as zero or as an error.

The showcase applies a strict standard: locations are retained only if they were observed in every year under consideration. This is a deliberate choice — persistence is only claimed when the full observation period corroborates it.


## Methodology

The overall workflow follows seven stages:

1. Perform Hot Spot Analysis (Getis-Ord Gi*) independently for each year.
2. Select one year as the spatial reference grid.
3. Compute the centroid of each analysis unit in the reference grid.
4. Spatially match each centroid against the corresponding analysis units from every other year.
5. Assemble a temporal profile containing one confidence value per year for each location.
6. Calculate summary statistics describing how consistently each location is classified as a statistically significant hot spot.
7. Retain and publish the locations that meet the desired persistence standard as a risk layer, ready for downstream use.

This approach transforms a series of independent annual analyses into a single, interpretable risk surface that reflects long-term accident risk rather than isolated events.

In [33]:
base_feature_class = feature_classes[0]
combined_df: pd.DataFrame = combine_yearly_hotspots(gdb_path, base_feature_class)
summarized_df: pd.DataFrame = summarize_hotspot_bins(combined_df, min_years_observed=len(feature_classes))
summarized_df

,OBJECTID,value,GiZScore,GiPValue,Gi_Bin,SHAPE,Gi_Bin_Traffic_Accidents_2019_Projected_HotSpots,Gi_Bin_Traffic_Accidents_2020_Projected_HotSpots,Gi_Bin_Traffic_Accidents_2021_Projected_HotSpots,Gi_Bin_Traffic_Accidents_2022_Projected_HotSpots,Gi_Bin_Traffic_Accidents_2023_Projected_HotSpots,Gi_Bin_Traffic_Accidents_2024_Projected_HotSpots,Gi_Bin_Traffic_Projected_HotSpots,Gi_Bin_median,Gi_Bin_years_observed,Gi_Bin_min,Gi_Bin_max
268,269,3.0,-0.025293,0.979821,-0.0,"{""rings"": [[[950283.7008999996, 6466434.863600...",-0.0,0.0,-0.0,0.0,-0.0,3.0,-0.0,-0.0,7,-0.0,3.0
271,272,4.0,1.078478,0.280821,0.0,"{""rings"": [[[950433.7008999996, 6464034.863600...",0.0,0.0,-0.0,0.0,-0.0,0.0,1.0,0.0,7,0.0,1.0
274,275,1.0,-1.387489,0.165293,-0.0,"{""rings"": [[[950433.7008999996, 6465684.863600...",-0.0,0.0,-0.0,0.0,3.0,0.0,-1.0,0.0,7,-1.0,3.0
282,283,2.0,2.135321,0.032735,2.0,"{""rings"": [[[950583.7008999996, 6463734.863600...",2.0,0.0,-0.0,0.0,-0.0,0.0,2.0,0.0,7,0.0,2.0
338,339,4.0,0.933584,0.350518,0.0,"{""rings"": [[[951483.7008999996, 6463884.863600...",0.0,0.0,-0.0,0.0,-0.0,0.0,1.0,0.0,7,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2200,2201,1.0,0.208514,0.834828,0.0,"{""rings"": [[[972933.7008999996, 6467484.863600...",0.0,0.0,-0.0,0.0,-0.0,3.0,-0.0,0.0,7,0.0,3.0
2202,2203,1.0,-0.561104,0.574727,-0.0,"{""rings"": [[[972933.7008999996, 6469284.863600...",-0.0,0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0,7,-0.0,-0.0
2258,2259,1.0,0.093106,0.925819,0.0,"{""rings"": [[[973533.7008999996, 6473934.863600...",0.0,0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0,7,0.0,0.0
2277,2278,2.0,0.093106,0.925819,0.0,"{""rings"": [[[973833.7008999996, 6473034.863600...",0.0,0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0,7,0.0,0.0


In [47]:
show_hotspot_map(home_portal, combined_df)

Map()

In [96]:
hottest_df = filter_hotspot_bins(summarized_df, min_value=2, max_value=3)
hottest_df.sort_values(by="Gi_Bin_median", ascending=False).head()

,OBJECTID,value,GiZScore,GiPValue,Gi_Bin,SHAPE,Gi_Bin_Traffic_Accidents_2019_Projected_HotSpots,Gi_Bin_Traffic_Accidents_2020_Projected_HotSpots,Gi_Bin_Traffic_Accidents_2021_Projected_HotSpots,Gi_Bin_Traffic_Accidents_2022_Projected_HotSpots,Gi_Bin_Traffic_Accidents_2023_Projected_HotSpots,Gi_Bin_Traffic_Accidents_2024_Projected_HotSpots,Gi_Bin_Traffic_Projected_HotSpots,Gi_Bin_median,Gi_Bin_years_observed,Gi_Bin_min,Gi_Bin_max
957,958,2.0,3.281122,0.001034,3.0,"{""rings"": [[[961233.7008999996, 6463884.863600...",3.0,1.0,3.0,3.0,-0.0,0.0,3.0,3.0,7,-0.0,3.0
1188,1189,6.0,4.553996,0.000005,3.0,"{""rings"": [[[963183.7008999996, 6464634.863600...",3.0,3.0,3.0,0.0,3.0,3.0,3.0,3.0,7,0.0,3.0
1168,1169,3.0,4.601287,0.000004,3.0,"{""rings"": [[[963033.7008999996, 6464634.863600...",3.0,3.0,3.0,0.0,3.0,3.0,3.0,3.0,7,0.0,3.0
1223,1224,7.0,3.074181,0.002111,3.0,"{""rings"": [[[963483.7008999996, 6464784.863600...",3.0,1.0,3.0,0.0,3.0,3.0,3.0,3.0,7,0.0,3.0
1280,1281,5.0,4.771831,0.000002,3.0,"{""rings"": [[[964083.7008999996, 6465234.863600...",3.0,0.0,3.0,3.0,3.0,2.0,3.0,3.0,7,0.0,3.0


## Key Questions This Session Answers

- Which locations exhibit persistent accident risk over multiple years?
- Which areas remain stable hotspots despite changing traffic conditions?
- Which locations warrant further investigation and mitigation measures?
- How can historical evidence support future decision-making?

In [9]:
existing_folder = "Spatial Data Science"
hotspot_item = publish_hotspot_layer(home_portal, hottest_df, title="Frankfurt Traffic Safety Hotspots", field_name="Gi_Bin_median", tags="DevSummit","Demo", folder=existing_folder)
hotspot_item

<Item title:"Frankfurt Traffic Safety Hotspots" type:Feature Layer Collection owner:devfromscratch>